# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmeer-Siddiqui125/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

This notebook defines and verifies the data contract for the Search Intelligence classification lane.

The goal is to classify content into higher- and lower-engagement groups using information that is available at the prediction decision moment.

Development and verification use the March 2026 warehouse partition (`2026-03`). The final month is treated as a sealed outcome/test window.

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET hf_secret "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face")

DuckDB connected to Hugging Face


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row represents the performance of **one content item for one client on one report date** in the daily performance warehouse.

The grain is therefore:

**`report_date × client_hash_id × content_hash_id`**

### Time window

I will use the **March 2026 panel (`2026-03`)** for development and verification.

March 2026 contains daily observations from **2026-03-01 through 2026-03-31**.

I use March rather than the final month because the final month is treated as a sealed outcome/test window.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
march_check = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").fetchone()

print("March row count:", march_check[0])
print("Date range:", march_check[1], "to", march_check[2])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March row count: 9841378
Date range: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The five candidate features for the engagement classification task are:

1. `gsc_impressions` — search visibility available before the decision moment.
2. `gsc_clicks` — search clicks available before the decision moment.
3. `gsc_avg_position` — average search position available before the decision moment.
4. `ga4_sessions` — sessions available before the decision moment when GA4 data is available.
5. `ga4_engaged_sessions` — engaged sessions available before the decision moment when GA4 data is available.

### Label / proxy

The prediction target is **future engagement level**.

The label will represent whether a content item has higher or lower engagement based on an engagement outcome measured after the feature observation period.

The label is not used as an input feature.

### Context

- `report_date` — identifies the observation date.
- `client_hash_id` — identifies the client and can be used for grouping or train/test splitting.
- `content_hash_id` — identifies the content item.
- `client_has_gsc` — indicates whether the client has GSC data.
- `client_has_ga4` — indicates whether the client has GA4 data.
- `gsc_data_available` — indicates whether GSC data is available for the observation.
- `ga4_data_available` — indicates whether GA4 data is available for the observation.

### Excluded

The following are deliberately excluded from the feature set:

- Label-derived engagement fields — they contain or are derived from the outcome being predicted and would cause target leakage.
- Future-period performance — information that would not be available at the prediction decision moment.
- `client_hash_id` and `content_hash_id` as model inputs — they are identifiers, not meaningful predictive measurements.
- The final June 2026 sample — it is reserved as a sealed outcome/test window rather than being used during development.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Fields selected for the engagement classification contract

features = [
    "gsc_impressions","gsc_clicks",
    "gsc_avg_position","ga4_sessions",
    "ga4_engaged_sessions"
]

label = "future_engagement"

context = [
    "report_date","client_hash_id",
    "content_hash_id","client_has_gsc",
    "client_has_ga4","gsc_data_available",
    "ga4_data_available"
]

excluded = [
    "label-derived fields",
    "future-period performance",
    "client_hash_id as a model feature",
    "content_hash_id as a model feature",
    "June 2026 final-month sample"
]

print("Features:")
for field in features:
    print("-", field)

print("\nLabel:")
print("-", label)

print("\nContext:")
for field in context:
    print("-", field)

print("\nExcluded:")
for field in excluded:
    print("-", field)

Features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

Label:
- future_engagement

Context:
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

Excluded:
- label-derived fields
- future-period performance
- client_hash_id as a model feature
- content_hash_id as a model feature
- June 2026 final-month sample


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The contract is verified using three small queries on the March 2026 panel.

1. Confirm the row count and date window.
2. Confirm the stated grain: `report_date × client_hash_id × content_hash_id`.
3. Confirm that performance data is only treated as available when the corresponding availability flag is `TRUE`.

The queries below are used to verify the contract rather than assuming the schema behaves as expected.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchall()

print("Duplicate grain rows:", grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows: []


In [5]:
availability_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").fetchone()

print("Total March rows:", availability_check[0])
print("Rows with GSC available:", availability_check[1])
print("Rows with GA4 available:", availability_check[2])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total March rows: 9841378
Rows with GSC available: 3611061
Rows with GA4 available: 413966


## Five-feature frame

To avoid target leakage, the prediction decision moment is set at **2026-03-15**.

The five features are calculated only from observations available from **2026-03-01 through 2026-03-15**.

The engagement outcome is measured after the decision moment, from **2026-03-16 through 2026-03-31**.

### Features and availability at decision time

| Feature | Why it is knowable at the decision moment |
|---|---|
| `gsc_impressions` | Search impressions recorded up to March 15 were already available when the decision was made. |
| `gsc_clicks` | Search clicks recorded up to March 15 were already available when the decision was made. |
| `gsc_avg_position` | Average search position observed up to March 15 was available before the prediction period. |
| `ga4_sessions` | GA4 sessions recorded up to March 15 were available when GA4 data was available. |
| `ga4_engaged_sessions` | GA4 engaged sessions recorded up to March 15 were available when GA4 data was available. |

The feature frame therefore contains only information that precedes the engagement outcome.

In [6]:
feature_frame = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS gsc_impressions,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS gsc_clicks,

        AVG(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS gsc_avg_position,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_sessions
                ELSE 0
            END
        ) AS ga4_sessions,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_engaged_sessions
                ELSE 0
            END
        ) AS ga4_engaged_sessions

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'

    GROUP BY
        client_hash_id,
        content_hash_id

    LIMIT 1000
""").fetchdf()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (1000, 7)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,2.0,4.247255,0.0,0.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,18.0,0.0,4.939394,0.0,0.0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,89.0,0.0,3.010741,0.0,0.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,1.0,5.330069,0.0,0.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,9.0,4.468441,0.0,0.0


### Future engagement label

The label represents engagement after the prediction decision moment.

For each client-content pair, future engagement rate is calculated over **2026-03-16 through 2026-03-31** as:

`future_engagement_rate = future_ga4_engaged_sessions / future_ga4_sessions`

Only observations where GA4 data is available and future sessions are greater than zero are used to calculate the rate.

The binary label is:

- `1` = future engagement rate at or above the median
- `0` = future engagement rate below the median

This label is calculated only from the future outcome window and is therefore not available at the March 15 decision moment.

In [7]:
future_label = con.execute("""
    WITH future_content AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_sessions
                    ELSE 0
                END
            ) AS future_sessions,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_engaged_sessions
                    ELSE 0
                END
            ) AS future_engaged_sessions

        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        )

        WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'

        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    engagement_rates AS (
        SELECT
            client_hash_id,
            content_hash_id,
            future_sessions,
            future_engaged_sessions,
            future_engaged_sessions * 1.0 / future_sessions
                AS future_engagement_rate
        FROM future_content
        WHERE future_sessions > 0
    ),

    positive_median AS (
        SELECT
            MEDIAN(future_engagement_rate) AS median_positive_rate
        FROM engagement_rates
        WHERE future_engagement_rate > 0
    )

    SELECT
        e.client_hash_id,
        e.content_hash_id,
        e.future_sessions,
        e.future_engaged_sessions,
        e.future_engagement_rate,

        CASE
            WHEN e.future_engagement_rate > 0
                 AND e.future_engagement_rate >= p.median_positive_rate
            THEN 1
            ELSE 0
        END AS future_engagement

    FROM engagement_rates e
    CROSS JOIN positive_median p

    LIMIT 1000
""").fetchdf()

print("Future label frame shape:", future_label.shape)

print(
    "Median positive future engagement rate:",
    future_label.loc[
        future_label["future_engagement_rate"] > 0,
        "future_engagement_rate"
    ].median()
)

print("\nLabel distribution:")
print(future_label["future_engagement"].value_counts())

display(future_label.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future label frame shape: (1000, 6)
Median positive future engagement rate: 0.19615384615384618

Label distribution:
future_engagement
0    787
1    213
Name: count, dtype: int64


,client_hash_id,content_hash_id,future_sessions,future_engaged_sessions,future_engagement_rate,future_engagement
0,client_3ffa76342f366962,content_81c69e96f7e779e0,1.0,0.0,0.0,0
1,client_3ffa76342f366962,content_eb28bd616c790db2,1.0,0.0,0.0,0
2,client_3ffa76342f366962,content_0e3f56b7ff47644d,1.0,0.0,0.0,0
3,client_3ffa76342f366962,content_bef14622c86fb662,1.0,0.0,0.0,0
4,client_3ffa76342f366962,content_3bf0a4d6ee00dfff,1.0,0.0,0.0,0


### Honest feature/label frame

The honest modeling frame combines features available before the decision moment with the future engagement label.

Features are calculated from **2026-03-01 through 2026-03-15**.

The label is calculated from **2026-03-16 through 2026-03-31**.

Therefore, none of the five features uses information from the future outcome period.

In [8]:
honest_frame = con.execute("""
    WITH features AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS gsc_impressions,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS gsc_clicks,

            AVG(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN gsc_avg_position
                    ELSE NULL
                END
            ) AS gsc_avg_position,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_sessions
                    ELSE 0
                END
            ) AS ga4_sessions,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_engaged_sessions
                    ELSE 0
                END
            ) AS ga4_engaged_sessions

        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        )

        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'

        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    future_content AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_sessions
                    ELSE 0
                END
            ) AS future_sessions,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_engaged_sessions
                    ELSE 0
                END
            ) AS future_engaged_sessions

        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        )

        WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'

        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    labeled AS (
        SELECT
            client_hash_id,
            content_hash_id,
            future_engaged_sessions * 1.0 / future_sessions
                AS future_engagement_rate
        FROM future_content
        WHERE future_sessions > 0
    ),

    threshold AS (
        SELECT
            MEDIAN(future_engagement_rate) AS median_rate
        FROM labeled
        WHERE future_engagement_rate > 0
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        f.ga4_sessions,
        f.ga4_engaged_sessions,

        CASE
            WHEN l.future_engagement_rate > 0
                 AND l.future_engagement_rate >= t.median_rate
            THEN 1
            ELSE 0
        END AS future_engagement

    FROM features f

    INNER JOIN labeled l
        ON f.client_hash_id = l.client_hash_id
        AND f.content_hash_id = l.content_hash_id

    CROSS JOIN threshold t

    LIMIT 1000
""").fetchdf()

print("Honest frame shape:", honest_frame.shape)

print("\nColumns:")
print(list(honest_frame.columns))

print("\nLabel distribution:")
print(honest_frame["future_engagement"].value_counts())

display(honest_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest frame shape: (1000, 8)

Columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'future_engagement']

Label distribution:
future_engagement
0    876
1    124
Name: count, dtype: int64


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,future_engagement
0,client_9958f0a7ae1df715,content_eb0aeedbcfaf2712,140.0,0.0,22.006968,2.0,0.0,0
1,client_9958f0a7ae1df715,content_4cec18f637b4c858,21.0,1.0,26.983333,7.0,0.0,0
2,client_9958f0a7ae1df715,content_6edb421d967970be,1376.0,1.0,7.787972,1.0,0.0,0
3,client_9958f0a7ae1df715,content_1886b144fd43edc5,930.0,1.0,6.075962,4.0,1.0,0
4,client_9958f0a7ae1df715,content_edb2dd3126f4e267,527.0,0.0,8.360961,4.0,0.0,0


## Deliberate label leakage trap

To demonstrate target leakage, I will deliberately add one column derived from the future engagement outcome.

`future_engagement_rate` is calculated from the same future period used to create `future_engagement`.

This column would not be available at the March 15 decision moment, so it is an invalid feature.

I will temporarily include it to demonstrate how leakage can produce an artificially strong model score. I will then remove it and retain the honest feature set.

In [9]:
leaky_frame = con.execute("""
    WITH future_content AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_sessions
                    ELSE 0
                END
            ) AS future_sessions,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN ga4_engaged_sessions
                    ELSE 0
                END
            ) AS future_engaged_sessions

        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        )

        WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'

        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    rates AS (
        SELECT
            client_hash_id,
            content_hash_id,
            future_engaged_sessions * 1.0 / future_sessions
                AS future_engagement_rate
        FROM future_content
        WHERE future_sessions > 0
    )

    SELECT
        h.*,
        r.future_engagement_rate
    FROM honest_frame h
    INNER JOIN rates r
        ON h.client_hash_id = r.client_hash_id
        AND h.content_hash_id = r.content_hash_id
""").fetchdf()

print("Leaky frame shape:", leaky_frame.shape)
print("Added leaked column: future_engagement_rate")

display(leaky_frame.head())

Leaky frame shape: (1000, 9)
Added leaked column: future_engagement_rate


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,future_engagement,future_engagement_rate
0,client_9958f0a7ae1df715,content_eb0aeedbcfaf2712,140.0,0.0,22.006968,2.0,0.0,0,0.0
1,client_9958f0a7ae1df715,content_4cec18f637b4c858,21.0,1.0,26.983333,7.0,0.0,0,0.0
2,client_9958f0a7ae1df715,content_6edb421d967970be,1376.0,1.0,7.787972,1.0,0.0,0,0.0
3,client_9958f0a7ae1df715,content_1886b144fd43edc5,930.0,1.0,6.075962,4.0,1.0,0,0.0
4,client_9958f0a7ae1df715,content_edb2dd3126f4e267,527.0,0.0,8.360961,4.0,0.0,0,0.0


## Leakage experiment

A simple Decision Tree classifier is used to compare the honest and leaky feature sets.

The honest model uses only the five features available at the decision moment.

The leaky model additionally uses `future_engagement_rate`, which is derived from the future outcome period.

A large increase in the leaky model's score demonstrates why label-derived fields must be excluded.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

# Remove rows with missing feature values for this small demonstration
honest_model_frame = honest_frame.dropna(
    subset=feature_cols + ["future_engagement"]
).copy()

X_honest = honest_model_frame[feature_cols]
y = honest_model_frame["future_engagement"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_f1 = f1_score(y_test, honest_pred)

print("Honest F1:", round(honest_f1, 4))

Honest F1: 0.0556


In [11]:
leaky_feature_cols = feature_cols + [
    "future_engagement_rate"
]

leaky_model_frame = leaky_frame.dropna(
    subset=leaky_feature_cols + ["future_engagement"]
).copy()

X_leaky = leaky_model_frame[leaky_feature_cols]
y_leaky = leaky_model_frame["future_engagement"]

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.25,
    random_state=42,
    stratify=y_leaky
)

leaky_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

leaky_model.fit(X_train_leak, y_train_leak)

leaky_pred = leaky_model.predict(X_test_leak)
leaky_f1 = f1_score(y_test_leak, leaky_pred, zero_division=0)

print("Leaky F1 score:", round(leaky_f1, 4))

Leaky F1 score: 1.0


In [12]:
print("Leakage experiment results")
print("-" * 35)
print("Honest F1:", round(honest_f1, 4))
print("Leaky F1: ", round(leaky_f1, 4))
print("-" * 35)
print("Score increase:", round(leaky_f1 - honest_f1, 4))

Leakage experiment results
-----------------------------------
Honest F1: 0.0556
Leaky F1:  1.0
-----------------------------------
Score increase: 0.9444


### Removing the leaked feature

The leakage experiment shows that `future_engagement_rate` can artificially improve the model score because it is calculated from the same future period used to create the target.

This feature would not be available at the March 15 decision moment. It is therefore removed from the final feature set.

The final honest feature set remains:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `ga4_engaged_sessions`

The honest score is retained as the valid result for this experiment. The higher leaky score is not treated as real model performance.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data has several limitations that affect the engagement classification task.

- **Uneven data availability:** GSC and GA4 information is not available for every observation. Features from these sources must therefore be used together with their availability flags rather than treating unavailable observations as genuine zero activity.

- **Different history depth:** Clients can have different amounts of historical data. A missing historical observation should not automatically be interpreted as zero performance.

- **Search and analytics coverage differs:** Some clients have usable GSC data while others may have limited or no usable search history. The same applies to GA4.

- **Future information cannot be used as a feature:** Any metric calculated from the outcome period must be excluded from the model inputs.

- **Window alignment matters:** Features must represent information available at the decision moment. A feature that overlaps with or is derived from the future engagement outcome would create leakage.

- **Identifiers are not predictive measurements:** `client_hash_id` and `content_hash_id` are used for grouping, joining, or splitting data, but are not used as model features.

### Named limitation

The main limitation is **uneven data availability across clients and dates**, which can reduce the amount of usable training data and may introduce differences in coverage between observations.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before constructing the feature frame, I confirm:

- [x] One row represents one client × content × report date.
- [x] March 2026 is the development window.
- [x] The March partition contains 9,841,378 rows.
- [x] The March date range is 2026-03-01 to 2026-03-31.
- [x] No duplicate rows were found at the stated grain.
- [x] GSC availability was checked using `IS TRUE`.
- [x] GA4 availability was checked using `IS TRUE`.
- [x] Client and content identifiers are treated as context rather than predictive features.
- [x] Future or label-derived information will be excluded from model inputs.
- [x] I defined one row as one client × content × report date.
- [x] I used March 2026 as the development window.
- [x] I verified the row count and date span.
- [x] I verified the dataset grain.
- [x] I checked availability using `IS TRUE`.
- [x] I selected five features.
- [x] I explained why each feature is available at the decision moment.
- [x] I created a future engagement proxy after the decision moment.
- [x] I separated the feature window from the outcome window.
- [x] I deliberately added a label-derived feature.
- [x] I compared the leaky score with the honest score.
- [x] I removed the leaked feature from the final feature set.
- [x] I identified uneven GSC and GA4 availability as a limitation.

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.